In [23]:
# Ensure the latest version of the module is imported
%load_ext autoreload
%autoreload 2
from multiband_fit import *
from collections.abc import Sequence
from eztaox.initializers import DRWInit, UniformInit, InitializerBase

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
objs = concat_light_curves(filter_object_ids=['1435352'])

Found 1 matching objects 1


Processing quasars: 100%|██████████| 1/1 [00:00<00:00,  2.06it/s]

Populating SDSS fields: 1



Populating SDSS fields: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]


In [24]:
class ExpPolyInit(InitializerBase):
    logScaleRange: Sequence[JAXArray | float]
    logSigmaRange: Sequence[JAXArray | float]
    logPolyScaleRange: Sequence[JAXArray | float]
    logPolySigmaRange: Sequence[JAXArray | float]

    def __init__(
        self,
        logScaleRange: Sequence[JAXArray | float],
        logSigmaRange: Sequence[JAXArray | float],
        logPolyScaleRange: Sequence[JAXArray | float],
        logPolySigmaRange: Sequence[JAXArray | float],
    ) -> None:
        self.logScaleRange = logScaleRange
        self.logSigmaRange = logSigmaRange
        self.logPolyScaleRange = logPolyScaleRange
        self.logPolySigmaRange = logPolySigmaRange

    def __call__(
        self,
        key: jax.random.PRNGKey,
        nSample: int,
    ) -> JAXArray:
        key1, key2, key3, key4 = jax.random.split(key, num=4)
        logScale = dist.Uniform(*self.logScaleRange).sample(key1, (nSample,))
        logSigma = dist.Uniform(*self.logSigmaRange).sample(key2, (nSample,))
        logPolyScale = dist.Uniform(*self.logPolyScaleRange).sample(key3, (nSample,))
        logPolySigma = dist.Uniform(*self.logPolySigmaRange).sample(key4, (nSample,))

        if nSample == 1:
            return jnp.stack([logScale, logSigma, logPolyScale, logPolySigma], axis=1)[0]
        else:
            return jnp.stack([logScale, logSigma, logPolyScale, logPolySigma], axis=-1)

class ExpPolyKernel(tinygp.kernels.Kernel):
    exp_amplitude: float
    exp_kernel: tinygp.kernels.Kernel
    poly_kernel: tinygp.kernels.Kernel

    def __init__(self, exp_scale, exp_amplitude, poly_scale, poly_amplitude):
        jax.debug.print("exp_scale {x}", x=exp_scale)
        jax.debug.print("exp_amplitude {x}", x=exp_amplitude)
        jax.debug.print("poly_scale {x}", x=poly_scale)
        jax.debug.print("poly_amplitude {x}", x=poly_amplitude)
        self.exp_amplitude = exp_amplitude
        self.exp_kernel = kernels.stationary.Exp(exp_scale)
        self.poly_kernel = kernels.Polynomial(1, poly_scale, poly_amplitude)

    def evaluate(self, X1, X2):
        return self.exp_amplitude * self.exp_kernel.evaluate(X1, X2) + self.poly_kernel.evaluate(X1, X2)


In [27]:
num_samples = 250

# define params
zero_mean = False
has_jitter = True
has_lag = True

lambda_pivot = {
    'u': 3543,  # SDSS u-band
    'g': 4770,  # SDSS g-band
    'r': 6231,  # SDSS r-band
    'i': 7625,  # SDSS i-band
    'z': 9134,  # SDSS z-band
    'y': 9633,  # PS1 y-band
}

filters = {"u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "y": 5} # harcoded filter order for SDSS
bands = ['u', 'g', 'r', 'i', 'z']#, 'y']

colors = {'u': 'tab:blue',
          'g': 'tab:green', 
          'r': 'tab:orange', 
          'i': 'tab:red', 
          'z': 'tab:brown', 
          'y': 'tab:gray'}

# Override MultiVarModel
class MyMultiVarModel(MultiVarModelFFT):
    filtered_bands: JAXArray

    def __init__(
        self,
        X: JAXArray,
        y: JAXArray | NDArray,
        yerr: JAXArray | NDArray,
        kernel: tinygp.kernels.quasisep.Quasisep,
        **kwargs,
    ) -> None:
        super().__init__(X, y, yerr, kernel, **kwargs)
        self.filtered_bands = kwargs.get("filtered_bands", None)

    def amp_transform(self, params: dict[str, JAXArray]) -> JAXArray:
        b = params["beta"]
        params["log_amp_delta"] = jnp.array([b*np.log(lambda_pivot[band]/lambda_pivot[self.filtered_bands[0]]) for band in self.filtered_bands[1:]]) # comment this out for old version
        r = jnp.insert(jnp.atleast_1d(params["log_amp_delta"]), 0, 0.0)
        return r
    pass
    
def initSampler(key, nSample, nBand=len(bands)):
    # split keys
    subkeys = jax.random.split(key, 10)

    # uniform sampler
    lagSampler = UniformInit(nBand-1, [-10, 10])
    meanSampler = UniformInit(nBand, [-1, 1])
    logAmpDeltaSampler = UniformInit(nBand-1, [-2, 0.0])
    logJitterSampler = UniformInit(nBand, [-20, -5])
    betaSampler = UniformInit(1, [-2.0, 0.0])

    # kernel init
    #kernelSampler = DRWInit([jnp.log(100), jnp.log(0.1)], [jnp.log(0.01), 0.0])
    kernelSampler = ExpPolyInit([jnp.log(100), jnp.log(0.1)], [jnp.log(0.01), 0.0],
                                [jnp.log(1000), jnp.log(0.1)], [jnp.log(0.01), 0.0])
    return {
        "log_kernel_param": kernelSampler(subkeys[0], nSample),
        "log_amp_delta": logAmpDeltaSampler(subkeys[1], nSample),
        "mean": meanSampler(subkeys[2], nSample),
        "lag": lagSampler(subkeys[3], nSample),
        "log_jitter": logJitterSampler(subkeys[4], nSample),
        "beta": betaSampler(subkeys[5], nSample),
    }
def numpyro_model(X, yerr, y=None, bestP=None, filtered_bands=None):
    # kernel param
    # flat_normal = dist.Normal(bestP["log_kernel_param"], jnp.array([5.0, 5.0]))
    flat_normal = dist.Normal(bestP["log_kernel_param"], jnp.array([5.0, 5.0, 5.0, 5.0]))
    diag_normal = dist.Independent(flat_normal, 1)
    log_kernel_param = numpyro.sample("log_kernel_param", diag_normal)

    # log amp delta
    #log_amp_delta = numpyro.sample(
    #   "log_amp_delta", dist.Normal(bestP["log_amp_delta"], 2.0)
    #) # comment this out when using beta

    # lag
    lag = numpyro.sample("lag", dist.Normal(bestP['lag'], 10.0))
    
    # log jitter, mean => the prior for these two should be set small, otherwise
    # it is hard to converge
    log_jitter = numpyro.sample("log_jitter", dist.Normal(bestP["log_jitter"], 0.1))
    
    mean = numpyro.sample("mean", dist.Normal(bestP['mean'], 0.1))

    beta = numpyro.sample("beta", dist.Normal(-0.5, 0.25))

    # kernel
    #k = kernels.quasisep.Exp(*jnp.exp(log_kernel_param))
    k = ExpPolyKernel(*jnp.exp(log_kernel_param))
    m1 = MyMultiVarModel(X, y, yerr, k, zero_mean=zero_mean, has_jitter=has_jitter, has_lag=has_lag, filtered_bands=filtered_bands)

    sample_params = {
        "log_kernel_param": log_kernel_param,
        #"log_amp_delta": log_amp_delta, # comment this out when using beta
        "lag": lag,
        "mean": mean,
        "log_jitter": log_jitter,
        "beta": beta,
    }
    m1.sample(sample_params)


def fit_multiband(data):
    times = data['times']
    mags = data['mags']
    magerrs = data['magerrs']

    # Drop bands that cross the Lyman break
    lyman_break_wavelength = 912  # in Angstroms
    rest_frame_wavelengths = {band: lambda_pivot[band] / (1 + data['z']) for band in bands}
    filtered_bands = [band for band in bands if rest_frame_wavelengths[band] > lyman_break_wavelength]

    # Combine
    all_times = np.concatenate([times[b] for b in filtered_bands])
    all_mags = np.concatenate([mags[b] for b in filtered_bands]) 
    all_magerrs = np.concatenate([magerrs[b] for b in filtered_bands])
    band_idx = np.concatenate([np.full(len(times[b]), i) for i, b in enumerate(filtered_bands)])

    if len(all_times) == 0 or len(all_mags) == 0 or len(all_magerrs) == 0:
        print(f"No magnitudes or errors for quasar {data['object_id']}, skipping.", flush=True)
        return None
    # Check for NaNs
    if np.all(~np.isfinite(all_times)) or np.all(~np.isfinite(all_mags)) or np.all(~np.isfinite(all_magerrs)):
        print(f"NaN values ({len(~np.isfinite(all_mags))}/{len(all_mags)}) found in data for quasar {data['object_id']}, skipping.", flush=True)
        return None

    # Sort in time
    sort_idx = np.argsort(all_times)
    all_times = all_times[sort_idx]
    all_mags = all_mags[sort_idx]
    all_magerrs = all_magerrs[sort_idx]
    band_idx = band_idx[sort_idx]

    # Mask NaNs
    mask = np.isfinite(all_mags)
    all_times = all_times[mask]
    all_mags = all_mags[mask]
    all_magerrs = all_magerrs[mask]
    band_idx = band_idx[mask]


    # Define X, y, yerr, t
    # X = (all_times, band_idx)
    X = (jnp.array(all_times)-jnp.min(all_times), jnp.array(band_idx))
    y = np.array(all_mags)
    yerr = np.array(all_magerrs)
    t = np.array(all_times)

    # Reject outliers in moving window
    window_size = 20
    mask_outlier = np.ones(len(y), dtype=bool)
    for i in range(len(y)):
        if i < window_size or i >= len(y) - window_size:
            continue
        window = y[i - window_size:i + window_size + 1]
        if jnp.abs(y[i] - jnp.nanmean(window)) > 3 * st.median_abs_deviation(window):
            mask_outlier[i] = False

    X = (jnp.array(all_times[mask_outlier]) - jnp.min(all_times[mask_outlier]), jnp.array(band_idx[mask_outlier]))
    y = jnp.array(y[mask_outlier])
    yerr = jnp.array(yerr[mask_outlier])
    t = jnp.array(t[mask_outlier])

    # define kernel
    # initial_drw_params = {"log_kernel_param": jnp.log(np.array([100.0, 0.35]))}
    # k = kernels.quasisep.Exp(*jnp.exp(initial_drw_params["log_kernel_param"]))
    initial_drw_params = {"log_kernel_param": jnp.log(np.array([100.0, 0.35, 1000.0, 0.15]))}
    k = ExpPolyKernel(*jnp.exp(initial_drw_params["log_kernel_param"]))
    
    # define model
    m1 = MyMultiVarModel(
        X, y, yerr, k, zero_mean=zero_mean, has_jitter=has_jitter, has_lag=has_lag, filtered_bands=filtered_bands
    )
    bestP, logProb = fit(model=m1, 
                        optimizer=optax.adam(learning_rate=0.1),
                        initSampler=initSampler,
                        prng_key=jax.random.PRNGKey(0),
                        nSample=10_000, nIter=2, nBest=5)

    for k in bestP.keys():
        bestP[k] += 1e-4 * np.random.randn(*bestP[k].shape) 

    nuts_kernel = NUTS(
        partial(numpyro_model, bestP=bestP, filtered_bands=filtered_bands),
        dense_mass=True,
        target_accept_prob=0.9,
        # adapt_step_size=True,
    )

    mcmc = MCMC(
        nuts_kernel,
        num_warmup=num_samples,
        num_samples=num_samples,
        num_chains=2,
        progress_bar=True,
    )

    mcmc.run(jax.random.PRNGKey(1), X, yerr, y=y)
    samples = mcmc.get_samples(group_by_chain=False)
    diagnostics = mcmc.get_extra_fields()
    if np.all(diagnostics['diverging']):
        print(f"Diverging MCMC for quasar {data['object_id']}, skipping.", flush=True)
        return None
    
    fig = save_combined_plot(samples, m1, X, y, yerr, band_idx[mask_outlier], 
                       data['object_id'], filtered_bands=filtered_bands)
    plt.show(fig)
    fig = plot_posterior(samples, data['object_id'])
    plt.show(fig)
    log_tau_RF = np.median(np.log10(np.exp(samples['log_kernel_param'][:, 0])/(1+data['z'])))


    lambda_ref = 2500 # Any reference wavelength
    lambda_pivot_RF = lambda_pivot[filtered_bands[0]]/(1 + data['z'])
    
    log_sigma_RF = np.median(np.log10(np.exp(samples['log_kernel_param'][:, 1] + samples['beta']*np.log(lambda_ref/lambda_pivot_RF))))
    # beta
    beta = np.median(samples['beta'])

    return dict(
        log_tau_RF=log_tau_RF,
        log_sigma_RF=log_sigma_RF,
        beta=beta,
        filtered_bands=filtered_bands,
    )

d = fit_multiband(objs[0])
d


exp_scale 100.00000000000004
exp_amplitude 0.3499999999999999
poly_scale 999.9999999999998
poly_amplitude 0.15


TypeError: cannot reshape array of shape (0,) (size 0) into shape () (size 1)